# DEMO: DỰ ĐOÁN LỖ HỔNG TỪ MỘT FILE SOLIDITY

# **1. CÀI COMPILER**

In [ ]:
import subprocess, stat, json, collections
from pathlib import Path

SOLC_VERSION = "0.4.26"
SOLC_PATH = Path("/content/solc") / f"solc-{SOLC_VERSION}"
SOLC_PATH.parent.mkdir(parents=True, exist_ok=True)

if not SOLC_PATH.exists():
    url = ("https://github.com/ethereum/solidity/releases/download/"
           f"v{SOLC_VERSION}/solc-static-linux")
    print("Đang tải solc", SOLC_VERSION, "...")
    subprocess.run(["wget", "-q", "-O", str(SOLC_PATH), url], check=True)
    SOLC_PATH.chmod(SOLC_PATH.stat().st_mode | stat.S_IEXEC)

print(subprocess.run([str(SOLC_PATH), "--version"],
                     capture_output=True, text=True).stdout.strip())

Đang tải solc 0.4.26 ...
solc, the solidity compiler commandline interface
Version: 0.4.26+commit.4563c3fc.Linux.g++


# **2. NẠP MÔ HÌNH**

In [ ]:
from google.colab import drive, files
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from tensorflow import keras

drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/IS6002/EXPERIMENT/implementation")
KMEANS_PATH     = BASE_DIR / "algorithm_training/KMeans/minibatch_kmeans.joblib"
AE_MODEL_PATH   = BASE_DIR / "algorithm_training/Autoencoder/autoencoder.keras"
AE_PACKAGE_PATH = BASE_DIR / "algorithm_training/Autoencoder/autoencoder_package.joblib"

km_pkg = joblib.load(KMEANS_PATH)
kmeans_model           = km_pkg["model"]
KM_VULNERABLE_CLUSTERS = set(km_pkg["vulnerable_clusters"])
KM_OFFSET              = km_pkg.get("cluster_id_offset", 1)

autoencoder  = keras.models.load_model(AE_MODEL_PATH)
ae_pkg       = joblib.load(AE_PACKAGE_PATH)
AE_SCALER    = ae_pkg["scaler"]
AE_THRESHOLD = float(ae_pkg["threshold"])

OPCODE_FEATURES = list(km_pkg["opcode_features"])
assert OPCODE_FEATURES == list(ae_pkg["opcode_features"])

print("========== MÔ HÌNH ==========")
print("K-Means     :", kmeans_model.n_clusters, "cụm | cụm Vulnerable:",
      sorted(KM_VULNERABLE_CLUSTERS))
print("Autoencoder : ngưỡng", f"{AE_THRESHOLD:.10f}")
print("Đặc trưng   :", len(OPCODE_FEATURES), "opcode")

Mounted at /content/drive
========== MÔ HÌNH ==========
K-Means     : 6 cụm | cụm Vulnerable: [1, 3, 4]
Autoencoder : ngưỡng 0.0000629042
Đặc trưng   : 256 opcode


# **3. PIPELINE XỬ LÝ**

In [ ]:
OPCODES = {
    0x00: "STOP", 0x01: "ADD", 0x02: "MUL", 0x03: "SUB", 0x04: "DIV",
    0x05: "SDIV", 0x06: "MOD", 0x07: "SMOD", 0x08: "ADDMOD", 0x09: "MULMOD",
    0x0a: "EXP", 0x0b: "SIGNEXTEND",
    0x10: "LT", 0x11: "GT", 0x12: "SLT", 0x13: "SGT", 0x14: "EQ",
    0x15: "ISZERO", 0x16: "AND", 0x17: "OR", 0x18: "XOR", 0x19: "NOT",
    0x1a: "BYTE", 0x1b: "SHL", 0x1c: "SHR", 0x1d: "SAR",
    0x20: "SHA3",
    0x30: "ADDRESS", 0x31: "BALANCE", 0x32: "ORIGIN", 0x33: "CALLER",
    0x34: "CALLVALUE", 0x35: "CALLDATALOAD", 0x36: "CALLDATASIZE",
    0x37: "CALLDATACOPY", 0x38: "CODESIZE", 0x39: "CODECOPY",
    0x3a: "GASPRICE", 0x3b: "EXTCODESIZE", 0x3c: "EXTCODECOPY",
    0x3d: "RETURNDATASIZE", 0x3e: "RETURNDATACOPY", 0x3f: "EXTCODEHASH",
    0x40: "BLOCKHASH", 0x41: "COINBASE", 0x42: "TIMESTAMP", 0x43: "NUMBER",
    0x44: "DIFFICULTY", 0x45: "GASLIMIT", 0x46: "CHAINID", 0x47: "SELFBALANCE",
    0x50: "POP", 0x51: "MLOAD", 0x52: "MSTORE", 0x53: "MSTORE8",
    0x54: "SLOAD", 0x55: "SSTORE", 0x56: "JUMP", 0x57: "JUMPI",
    0x58: "GETPC", 0x59: "MSIZE", 0x5a: "GAS", 0x5b: "JUMPDEST",
    0xf0: "CREATE", 0xf1: "CALL", 0xf2: "CALLCODE", 0xf3: "RETURN",
    0xf4: "DELEGATECALL", 0xf5: "CREATE2",
    0xfa: "STATICCALL", 0xfd: "REVERT", 0xfe: "INVALID", 0xff: "SELFDESTRUCT",
}
for _i in range(1, 33):
    OPCODES[0x5f + _i] = "PUSH%d" % _i
for _i in range(1, 17):
    OPCODES[0x7f + _i] = "DUP%d" % _i
    OPCODES[0x8f + _i] = "SWAP%d" % _i
for _i in range(0, 5):
    OPCODES[0xa0 + _i] = "LOG%d" % _i


def disassemble(bytecode_hex):
    """Bytecode hex -> danh sách tên opcode. Không đếm toán hạng của PUSH."""
    h = bytecode_hex.lower()
    if h.startswith("0x"):
        h = h[2:]
    if len(h) % 2 == 1:
        h = h[:-1]
    raw = bytes.fromhex(h)

    ops, i, n = [], 0, len(raw)
    while i < n:
        b = raw[i]
        name = OPCODES.get(b)
        if name is None:
            ops.append("OP_%02X" % b)
            i += 1
            continue
        ops.append(name)
        i += 1 + (b - 0x5f if 0x60 <= b <= 0x7f else 0)
    return ops


def strip_metadata(bytecode_hex):
    """Cắt vùng CBOR metadata ở cuối runtime bytecode."""
    h = bytecode_hex[2:] if bytecode_hex.lower().startswith("0x") else bytecode_hex
    if len(h) < 4:
        return h
    try:
        cbor_len = int(h[-4:], 16) * 2
    except ValueError:
        return h
    return h[: len(h) - 4 - cbor_len] if 0 < cbor_len < len(h) - 4 else h


def extract_opcodes(sol_path):
    """File .sol -> (Counter opcode, số contract đã dùng). Không truyền --optimize."""
    proc = subprocess.run(
        [str(SOLC_PATH), "--combined-json", "bin-runtime", str(sol_path)],
        capture_output=True, text=True, timeout=300
    )
    if proc.returncode != 0:
        raise RuntimeError("Biên dịch lỗi:\n" + proc.stderr.strip()[-700:])

    total, n_used = collections.Counter(), 0
    for item in json.loads(proc.stdout).get("contracts", {}).values():
        rt = item.get("bin-runtime", "")
        if not rt or set(rt) == {"0"}:
            continue
        total.update(disassemble(strip_metadata(rt)))
        n_used += 1

    if not total:
        raise RuntimeError("File không chứa contract nào có runtime bytecode.")
    return total, n_used


def true_label_from_name(filename):
    """Đọc nhãn thật từ tên file: *_NonVul -> an toàn, *_Vul -> có lỗ hổng."""
    s = Path(filename).stem.upper()
    if "NONVUL" in s:
        return "Non-vulnerable"
    if "VUL" in s:
        return "Vulnerable"
    return None


def predict_one(sol_path):
    """Chạy toàn bộ pipeline trên một file .sol và trả về kết quả."""
    counts, n_used = extract_opcodes(sol_path)

    # vector 256 chiều, đúng thứ tự cột của mô hình
    x = np.array([[counts.get(name, 0) for name in OPCODE_FEATURES]], dtype="float64")
    outside = [k for k in counts if k not in set(OPCODE_FEATURES)]

    # --- K-Means: log1p rồi chia theo hàng ---
    x_log = np.log1p(x)
    x_km = x_log / x_log.sum(axis=1, keepdims=True)
    cluster = int(kmeans_model.predict(x_km)[0]) + KM_OFFSET
    km_label = "Vulnerable" if cluster in KM_VULNERABLE_CLUSTERS else "Non-vulnerable"

    # --- Autoencoder: log1p rồi MinMaxScaler đã lưu, clip về [0,1] ---
    z = AE_SCALER.transform(x_log)
    clip_rate = float(((z < 0) | (z > 1)).mean())
    x_ae = np.clip(z, 0.0, 1.0).astype(np.float32)
    err = float(np.mean(np.square(x_ae - autoencoder.predict(x_ae, verbose=0))))
    ae_label = "Vulnerable" if err > AE_THRESHOLD else "Non-vulnerable"

    return dict(file=Path(sol_path).name,
                true_label=true_label_from_name(sol_path),
                km_label=km_label, cluster=cluster,
                ae_label=ae_label, ae_error=err,
                total_opcodes=int(x.sum()), n_distinct=len(counts),
                n_contracts=n_used, clip_rate=clip_rate,
                n_outside_vocab=len(outside))


print("Pipeline đã sẵn sàng.")

Pipeline đã sẵn sàng.


**KMEANS CHIA DỮ LIỆU LÀM 6 CỤM: CÁCH GÁN NHÃN CHO CỤM SẼ DỰA TRÊN SỐ LƯỢNG MẪU CÓ NHÃN CHIẾM ĐA SỐ**

**AUTOENCODER DỰA VÀO KẾT QUẢ TÁI TẠO VÀ SO SÁNH VỚI NGƯỠNG, NẾU SAI SỐ GIỮA ĐẦU RA VÀ ĐẦU VÀO LỚN HƠN NGƯỠNG -> CÓ LỖ HỔNG**

# **4. UPLOAD HỢP ĐỒNG VÀ DỰ ĐOÁN**

In [ ]:
uploaded = files.upload()

# Lưu kết quả dự đoán của tất cả hợp đồng
prediction_results = []

if not uploaded:
    print("Chưa chọn file nào.")

for filename in uploaded:
    path = Path("/content") / filename
    path.write_bytes(uploaded[filename])

    print("\n" + "=" * 58)
    print(f"FILE: {filename}")
    print("=" * 58)

    try:
        r = predict_one(path)
    except RuntimeError as e:
        print("KHÔNG XỬ LÝ ĐƯỢC")
        print(e)
        print("\nNguyên nhân thường gặp: hợp đồng viết bằng cú pháp Solidity 0.5+,")
        print("trong khi mô hình chỉ huấn luyện trên bytecode của solc 0.4.x.")
        continue

    # Lưu kết quả để tính Confusion Matrix
    prediction_results.append(r)

    print(f"  Cụm K-Means            : {r['cluster']}")
    print()
    print(f"  Nhãn thật              : {r['true_label'] or 'không rõ'}")
    print(f"  K-Means dự đoán        : {r['km_label']}")
    # So sánh Reconstruction Error với Threshold
    print()
    print("  ===== AUTOENCODER =====")
    print(f"  Reconstruction Error   : {r['ae_error']:.10f}")
    print(f"  Threshold              : {AE_THRESHOLD:.10f}")

    if r["ae_error"] > AE_THRESHOLD:
        print("  So sánh                : RE > Threshold")
        print("  Kết luận               : Vulnerable")
    else:
        print("  So sánh                : RE <= Threshold")
        print("  Kết luận               : Non-vulnerable")

print(f"\nĐã dự đoán thành công {len(prediction_results)} hợp đồng.")

Saving 2_Vul.sol to 2_Vul (5).sol

FILE: 2_Vul (5).sol
  Cụm K-Means            : 4

  Nhãn thật              : Vulnerable
  K-Means dự đoán        : Vulnerable

  ===== AUTOENCODER =====
  Reconstruction Error   : 0.0008938918
  Threshold              : 0.0000629042
  So sánh                : RE > Threshold
  Kết luận               : Vulnerable

Đã dự đoán thành công 1 hợp đồng.
